In [1]:
from pyspark.sql.functions import max, min, avg, count, round

In [ ]:
catalog = dbutils.widgets.get("catalog")

In [ ]:
df = spark.read.table(f"{catalog}.02_silver.jc_citibike")

In [3]:
df.show()

+----------------+---------------+--------------------+--------------------+------------------+--------------------+------------------+--------------------+
|         ride_id|trip_start_date|          started_at|            ended_at|start_station_name|    end_station_name|trip_duration_mins|            metadata|
+----------------+---------------+--------------------+--------------------+------------------+--------------------+------------------+--------------------+
|29DAF43DD84B4B7A|     2025-03-20|2025-03-20 18:58:...|2025-03-20 19:00:...|   6 St & Grand St|Mama Johnson Fiel...|              2.25|{pipeline_id -> p...|
|B11B4220F7195025|     2025-03-29|2025-03-29 11:01:...|2025-03-29 11:11:...|  Heights Elevator|        Jersey & 3rd| 9.733333333333333|{pipeline_id -> p...|
|18D5B30305F602B9|     2025-03-01|2025-03-01 16:05:...|2025-03-01 16:07:...|      Jersey & 3rd|       Hamilton Park| 2.183333333333333|{pipeline_id -> p...|
|532EB2D9DB68567D|     2025-03-21|2025-03-21 18:44:...|202

In [4]:

df = df.groupBy("trip_start_date").agg(
    round(max("trip_duration_mins"),2).alias("max_trip_duration_mins"),
    round(min("trip_duration_mins"),2).alias("min_trip_duration_mins"),
    round(avg("trip_duration_mins"),2).alias("avg_trip_duration_mins"),
    count("ride_id").alias("total_trips")
)

In [5]:
df.show()

+---------------+----------------------+----------------------+----------------------+-----------+
|trip_start_date|max_trip_duration_mins|min_trip_duration_mins|avg_trip_duration_mins|total_trips|
+---------------+----------------------+----------------------+----------------------+-----------+
|     2025-03-23|                559.77|                  1.03|                 10.17|       1875|
|     2025-03-16|                202.63|                  1.07|                  9.43|       1928|
|     2025-03-03|                 138.7|                  1.02|                  6.53|       2030|
|     2025-03-08|                  91.6|                  1.02|                  7.35|       2008|
|     2025-02-28|                 25.25|                   2.8|                  11.4|          4|
|     2025-03-07|               1499.93|                  1.03|                  8.21|       2338|
|     2025-03-12|                951.33|                  1.02|                  7.46|       2687|
|     2025

In [ ]:
df.write.\
    mode("overwrite").\
    option("overwriteSchema", "true").\
    saveAsTable(f"{catalog}.03_gold.daily_ride_summary")